# Calendar-Aware Scheduling

This notebook demonstrates how `trueppm-scheduler` handles real-world calendars:
non-standard working weeks, public holiday exceptions, and per-project timezone
configuration.

**Install**
```bash
pip install trueppm-scheduler
```

In [ ]:
from datetime import date, timedelta
from trueppm_scheduler import (
    Calendar, DateRange, Dependency, DependencyType,
    Project, Task, schedule,
)

## 1. The default calendar

`Calendar()` with no arguments gives a standard Mon–Fri, 8 h/day calendar in UTC.
The `working_days` field is a 7-bit mask: bit 0 = Monday … bit 6 = Sunday.

In [ ]:
default_cal = Calendar()
print(f"working_days bitmask: {default_cal.working_days:07b}  (Mon=bit0, Sun=bit6)")
print(f"hours_per_day:        {default_cal.hours_per_day}")
print(f"timezone:             {default_cal.timezone}")
print()
# Check which days are working
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
for i, name in enumerate(day_names):
    working = bool((default_cal.working_days >> i) & 1)
    print(f"  {name}: {'working' if working else 'non-working'}")

## 2. Custom working week: Saturday schedule

Some industries (construction, oil & gas) work Mon–Sat.
Set bit 5 (Saturday) in the `working_days` mask.

In [ ]:
# Mon–Sat: bits 0–5 = 0b0111111 = 63
mon_sat_cal = Calendar(working_days=0b0111111)

tasks = [
    Task(id="t1", name="Excavation",   duration=timedelta(days=10)),
    Task(id="t2", name="Foundation",   duration=timedelta(days=5)),
    Task(id="t3", name="Framing",      duration=timedelta(days=8)),
]
deps = [
    Dependency(predecessor_id="t1", successor_id="t2"),
    Dependency(predecessor_id="t2", successor_id="t3"),
]

def make_project(cal: Calendar, name: str) -> Project:
    return Project(
        id=name.lower().replace(" ", "-"),
        name=name,
        start_date=date(2026, 5, 4),  # Monday
        tasks=tasks,
        dependencies=deps,
        calendar=cal,
    )

r_5day = schedule(make_project(Calendar(),         "Mon–Fri"))
r_6day = schedule(make_project(mon_sat_cal, "Mon–Sat"))

print(f"Mon–Fri finish: {r_5day.project_finish}")
print(f"Mon–Sat finish: {r_6day.project_finish}")
print(f"Calendar saves {(r_5day.project_finish - r_6day.project_finish).days} calendar days")

## 3. Public holiday exceptions

Pass a list of `DateRange` objects to block out holidays.
The scheduler skips non-working days during the forward pass,
so a holiday mid-task pushes the finish date without any manual adjustment.

In [ ]:
# UK bank holidays in May 2026
uk_may_bank_holidays = [
    DateRange(start=date(2026, 5, 4), end=date(2026, 5, 4)),   # Early May BH
    DateRange(start=date(2026, 5, 25), end=date(2026, 5, 25)), # Spring BH
]

uk_cal = Calendar(exceptions=uk_may_bank_holidays)

# A single 20-working-day task starting on the first bank holiday
long_task = [Task(id="p", name="Phase 1", duration=timedelta(days=20))]

p_no_holiday = Project(
    id="no-holidays", name="No holidays",
    start_date=date(2026, 5, 4),
    tasks=long_task, dependencies=[], calendar=Calendar(),
)
p_uk = Project(
    id="uk-holidays", name="UK holidays",
    start_date=date(2026, 5, 4),
    tasks=long_task, dependencies=[], calendar=uk_cal,
)

r_no = schedule(p_no_holiday)
r_uk = schedule(p_uk)

for t in r_no.tasks:
    print(f"No holidays:   {t.early_start} → {t.early_finish}")
for t in r_uk.tasks:
    print(f"UK May BH:     {t.early_start} → {t.early_finish}")
print(f"Holiday slip: +{(r_uk.project_finish - r_no.project_finish).days} calendar days")

## 4. Multi-week holiday block

`DateRange.start` and `DateRange.end` are inclusive, so a two-week
Christmas shutdown is expressed as a single range.

In [ ]:
xmas_shutdown = Calendar(
    exceptions=[
        DateRange(start=date(2026, 12, 21), end=date(2027, 1, 4)),
    ]
)

year_end_tasks = [
    Task(id="qa",      name="QA",         duration=timedelta(days=10)),
    Task(id="uat",     name="UAT",        duration=timedelta(days=5)),
    Task(id="deploy",  name="Deploy",     duration=timedelta(days=2)),
]
year_end_deps = [
    Dependency(predecessor_id="qa",  successor_id="uat"),
    Dependency(predecessor_id="uat", successor_id="deploy"),
]

p_before_xmas = Project(
    id="before-xmas", name="Before Xmas",
    start_date=date(2026, 12, 10),
    tasks=year_end_tasks, dependencies=year_end_deps, calendar=xmas_shutdown,
)

r = schedule(p_before_xmas)
print(f"Project finish (with Xmas shutdown): {r.project_finish}")
print()
for t in r.tasks:
    print(f"  {t.name:<10} {t.early_start} → {t.early_finish}")

## 5. Calendar-aware lag on dependencies

Lag in `Dependency` is measured in **working days** on the project calendar.
A 5-day lag spanning a holiday block is automatically extended.

In [ ]:
bank_holiday_mid_lag = Calendar(
    exceptions=[
        DateRange(start=date(2026, 5, 25), end=date(2026, 5, 25)),  # Spring BH
    ]
)

t_a = Task(id="a", name="Kick-off",    duration=timedelta(days=1))
t_b = Task(id="b", name="Next phase",  duration=timedelta(days=5))

# 5-working-day mandatory wait after kick-off
dep_with_lag = Dependency(
    predecessor_id="a",
    successor_id="b",
    dep_type=DependencyType.FS,
    lag=timedelta(days=5),
)

p_with_bh    = Project(id="lag-bh",    name="Lag with BH",
                       start_date=date(2026, 5, 19),  # Tuesday before BH week
                       tasks=[t_a, t_b], dependencies=[dep_with_lag],
                       calendar=bank_holiday_mid_lag)
p_without_bh = Project(id="lag-no-bh", name="Lag no BH",
                       start_date=date(2026, 5, 19),
                       tasks=[t_a, t_b], dependencies=[dep_with_lag],
                       calendar=Calendar())

for project, label in [(p_without_bh, "No BH"), (p_with_bh, "With BH")]:
    r = schedule(project)
    b = next(t for t in r.tasks if t.id == "b")
    print(f"{label:10} → 'Next phase' starts {b.early_start}")

## 6. Verifying calendar JSON round-trip

Calendars are serialised as part of `Project.to_json()` and restored faithfully
by `Project.from_json()`. This matters for the Django API layer: projects are
serialised at dispatch time and the pure scheduler sees no database objects.

In [ ]:
import json

project_with_holidays = Project(
    id="rt-test", name="Round-trip test",
    start_date=date(2026, 5, 4),
    tasks=tasks, dependencies=deps,
    calendar=uk_cal,
)

json_str  = project_with_holidays.to_json(indent=2)
restored  = Project.from_json(json_str)

r_orig    = schedule(project_with_holidays)
r_rt      = schedule(restored)

assert r_orig.project_finish == r_rt.project_finish, "Round-trip mismatch!"
print("Round-trip OK:", r_rt.project_finish)

cal_data = json.loads(json_str)["calendar"]
print(f"Serialised exceptions: {cal_data['exceptions']}")

## 7. Integrating with Django: calendar from database

In the TruePPM Django app the scheduler is invoked from a Celery task.
The `Calendar` object is built from the `Project.calendar` FK before
the project is serialised and handed to the pure Python engine.

```python
# packages/api/src/trueppm_api/apps/scheduling/services.py (simplified)
from trueppm_scheduler import Calendar, DateRange, Project as SchedulerProject

def build_scheduler_calendar(django_project) -> Calendar:
    cal = django_project.calendar
    return Calendar(
        working_days=cal.working_days,
        hours_per_day=cal.hours_per_day,
        timezone=cal.timezone,
        exceptions=[
            DateRange(start=exc.start_date, end=exc.end_date)
            for exc in cal.exceptions.all()
        ],
    )
```

The scheduler never touches Django models directly — the service layer
handles the translation.

See `docs/integration/django.md` for a complete integration walkthrough.